In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../data/spam.csv', encoding='latin-1')

In [3]:
df = df.dropna(how="any", axis=1)
df.columns = ['label', 'text']

In [4]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [6]:
print(f"Total rows: {len(df)}")
df.head(20)

Total rows: 5572


,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
5,1,FreeMsg Hey there darling it's been 3 week's n...
6,0,Even my brother is not like to speak with me. ...
7,0,As per your request 'Melle Melle (Oru Minnamin...
8,1,WINNER!! As a valued network customer you have...
9,1,Had your mobile 11 months or more? U R entitle...


### EDA

In [9]:
print("Class Distribution")
print(df['label'].value_counts(normalize=True))

Class Distribution
label
0    0.865937
1    0.134063
Name: proportion, dtype: float64


In [10]:
df['message_length'] = df['text'].apply(len)

print("Average Message Length")
print(df.groupby('label')['message_length'].mean())

Average Message Length
label
0     71.023627
1    138.866131
Name: message_length, dtype: float64


In [11]:
print("Summary Statistics of Length")
print(df.groupby('label')['message_length'].describe())

Summary Statistics of Length
        count        mean        std   min    25%    50%    75%    max
label                                                                 
0      4825.0   71.023627  58.016023   2.0   33.0   52.0   92.0  910.0
1       747.0  138.866131  29.183082  13.0  132.5  149.0  157.0  224.0


In [12]:
# How many chars and wods in each message
df['char_count'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

# Compare statistics across classes
print("Descriptive Stats for Legitimate Messages (Ham)")
print(df[df['label'] == 0][['char_count', 'word_count']].describe().round(1))

print("Descriptive Stats for Spam Messages")
print(df[df['label'] == 1][['char_count', 'word_count']].describe().round(1))

Descriptive Stats for Legitimate Messages (Ham)
       char_count  word_count
count      4825.0      4825.0
mean         71.0        14.2
std          58.0        11.4
min           2.0         1.0
25%          33.0         7.0
50%          52.0        11.0
75%          92.0        19.0
max         910.0       171.0
Descriptive Stats for Spam Messages
       char_count  word_count
count       747.0       747.0
mean        138.9        23.9
std          29.2         5.8
min          13.0         2.0
25%         132.5        22.0
50%         149.0        25.0
75%         157.0        28.0
max         224.0        35.0


### Feature Engineering

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label'])

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 4457
Testing samples: 1115


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

Vocabulary size: 5000


### Logistic Regression

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

model = LogisticRegression(solver='liblinear')
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

Accuracy: 0.9686

Classification Report:
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       966
        Spam       0.99      0.77      0.87       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115



### Final results:

1. Accuracy 96.86% 
2. 99% of messages that model point as not spam - true positive
3. 77% spam was only detected, however it tradeoff, and can be imporved by tunning threshold


In [16]:
import joblib

joblib.dump(model, 'model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')

print("Artifacts successfully saved inside the 'models/' directory!")

Artifacts successfully saved inside the 'models/' directory!
